# BRB-seq shallow versus deep one-feature QC

This is a small pooling-QC notebook. It compares STAR one-feature, non-UMI-deduplicated gene counts from the shallow and deep runs, shows the count distributions, and identifies samples below the shallow-sequencing cutoff.

In [ ]:
library(Matrix)
library(ggplot2)

repo_root <- "/scratch/ztobias/Gacu_BC_txmics"
star_root <- file.path(repo_root, "outputs", "star")
metadata_root <- file.path(repo_root, "metadata")
shallow_cutoff <- 450

manifest <- read.delim(file.path(metadata_root, "brb_sample_barcodes.tsv"),
  stringsAsFactors = FALSE, check.names = FALSE)


In [ ]:
# Build a barcode map from the source metadata, including prepared samples
# that were later removed from the final pooling configuration.
norm_pos <- function(x) {
  x <- toupper(trimws(as.character(x)))
  out <- rep(NA_character_, length(x))
  ok <- grepl("^[A-P]0*[0-9]+$", x)
  out[ok] <- sprintf("%s%02d", substr(x[ok], 1, 1),
    as.integer(sub("^[A-P]0*", "", x[ok])))
  out
}

position_barcodes <- function(path) {
  x <- read.delim(path, stringsAsFactors = FALSE, check.names = FALSE)
  data.frame(position = norm_pos(x$Name), barcode = toupper(x$B1),
    stringsAsFactors = FALSE)
}

v5b <- position_barcodes(file.path(metadata_root, "barcodes_v5B_384_brb.txt"))
v5d <- position_barcodes(file.path(metadata_root, "barcodes_v5D_384_brb.txt"))

rna24 <- read.csv(file.path(metadata_root, "2024_BC_RNA_extractions.csv"),
  stringsAsFactors = FALSE, check.names = FALSE)
map24 <- rna24[rna24$`use?` == 1, c("sample", "tissue", "V5B well")]
map24$sample <- sub("_2$", "", trimws(map24$sample))
map24$year <- 2024
map24$position <- norm_pos(map24$`V5B well`)
map24$barcode <- v5b$barcode[match(map24$position, v5b$position)]
map24$tissue <- tolower(trimws(map24$tissue))
map24 <- map24[nzchar(map24$sample) & map24$tissue %in% c("gill", "liver") & !is.na(map24$position), ]
map24 <- map24[, c("sample", "year", "tissue", "position", "barcode")]

m25 <- read.csv(file.path(metadata_root, "BC Spring 2025 Collections and Molecular Metadata Sheet 1.csv"),
  stringsAsFactors = FALSE, check.names = FALSE)
names(m25)[1] <- "sample"
map25 <- rbind(
  data.frame(sample = trimws(m25$sample), year = 2025, tissue = "gill",
    position = norm_pos(m25$`gill rxn plate pos`), stringsAsFactors = FALSE),
  data.frame(sample = trimws(m25$sample), year = 2025, tissue = "liver",
    position = norm_pos(m25$`liver rxn plate pos`), stringsAsFactors = FALSE)
)
map25$barcode <- v5d$barcode[match(map25$position, v5d$position)]
map25 <- map25[nzchar(map25$sample) & nzchar(map25$position), ]
barcode_map <- rbind(map24, map25)
barcode_map <- barcode_map[!duplicated(barcode_map[, c("year", "tissue", "barcode")]), ]


In [ ]:
read_star_totals <- function(prefix, stage) {
  suffix <- if (stage == "shallow") "shallow" else "deep_1f"
  raw <- file.path(star_root, paste0(prefix, "_", suffix, "_Solo.out"), "Gene", "raw")
  matrix_file <- file.path(raw, "umiDedup-NoDedup.mtx")
  if (!file.exists(matrix_file)) stop("Missing STAR matrix: ", matrix_file)
  x <- readMM(matrix_file)
  barcodes <- readLines(file.path(raw, "barcodes.tsv"))
  if (ncol(x) != length(barcodes)) stop("Matrix/barcode length mismatch: ", raw)
  data.frame(barcode = barcodes, total_counts = as.numeric(colSums(x)),
    detected_genes = as.numeric(colSums(x > 0)), stringsAsFactors = FALSE)
}

one_run <- function(year, tissue) {
  prefix <- paste(tissue, year, sep = "_")
  shallow <- read_star_totals(prefix, "shallow")
  deep <- read_star_totals(prefix, "deep")
  out <- merge(shallow, deep, by = "barcode", suffixes = c("_shallow", "_deep"))
  out$year <- year; out$tissue <- tissue
  bm <- barcode_map[barcode_map$year == year & barcode_map$tissue == tissue, ]
  hit <- match(out$barcode, bm$barcode)
  out$sample <- bm$sample[hit]
  out$position <- bm$position[hit]
  out$final_pool_member <- out$sample %in% manifest$sample_id[manifest$year == year & manifest$tissue == tissue]
  out$below_shallow_cutoff <- out$total_counts_shallow < shallow_cutoff
  out
}

qc <- do.call(rbind, lapply(c(2024, 2025), function(y)
  do.call(rbind, lapply(c("gill", "liver"), function(t) one_run(y, t)))))
qc <- qc[!is.na(qc$sample), ]
rownames(qc) <- NULL
head(qc)


In [ ]:
# The shallow and deep distributions. The dashed line is the old pooling cutoff.
ggplot(qc, aes(x = total_counts_shallow, fill = factor(year))) +
  geom_histogram(bins = 50, na.rm = TRUE) +
  geom_vline(xintercept = shallow_cutoff, linetype = "dashed") +
  scale_x_log10() + facet_grid(tissue ~ ., scales = "free_y") +
  theme_classic() + labs(x = "Shallow one-feature counts (non-UMI)", fill = "Year")

ggplot(qc, aes(x = total_counts_shallow, y = total_counts_deep, color = factor(year))) +
  geom_point(na.rm = TRUE) +
  geom_vline(xintercept = shallow_cutoff, linetype = "dashed") +
  scale_x_log10() + scale_y_log10() + facet_wrap(~ tissue) +
  theme_classic() + labs(x = "Shallow counts", y = "Deep one-feature counts", color = "Year")


In [ ]:
# Samples below the shallow cutoff, with final-pool status.
drop_candidates <- qc[qc$below_shallow_cutoff | !qc$final_pool_member,
  c("sample", "year", "tissue", "barcode",
    "total_counts_shallow", "total_counts_deep",
    "detected_genes_shallow", "final_pool_member",
    "below_shallow_cutoff") ]
drop_candidates[order(drop_candidates$tissue, drop_candidates$year,
  drop_candidates$total_counts_shallow), ]

# For the confirmed 2025 gill exclusions, this should recover the seven
# positions documented in metadata/README.md.
subset(drop_candidates, year == 2025 & tissue == "gill" & !final_pool_member)
